# Multimodal Retrieval-Augmented Generation (RAG) with CAMEL

This cookbook demonstrates how to build a multimodal RAG pipeline using CAMEL.
Unlike traditional text-only RAG, multimodal RAG can retrieve and reason over
images, diagrams, and text together.

## What you'll learn
- Load and process images with CAMEL loaders
- Set up a hybrid retriever for text + image context
- Use a multimodal model to answer questions with visual context

## Prerequisites

```bash
pip install camel-ai
```

Set your API keys:

```python
import os
os.environ["OPENAI_API_KEY"] = "your-key-here"
```

In [ ]:
import os
from camel.loaders import ImageLoader, UrllibLoader
from camel.retrievers import HybridRetriever
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from camel.agents import ChatAgent

os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

## Step 1: Load multimodal data

Load both text documents and images using CAMEL's built-in loaders.

In [ ]:
# Load a text document from a URL
text_loader = UrllibLoader()
text_content = text_loader.load_file(
    "https://en.wikipedia.org/wiki/King_Abdullah_University_of_Science_and_Technology"
)
print(f"Loaded {len(text_content)} characters of text")

# Load an image
image_loader = ImageLoader()
image = image_loader.load(
    "https://upload.wikimedia.org/wikipedia/en/thumb/4/4e/KAUST_logo.svg/1200px-KAUST_logo.svg.png"
)
print(f"Loaded image: {image}")

## Step 2: Set up a hybrid retriever

The `HybridRetriever` combines vector search with keyword (BM25) search
for more accurate retrieval.

In [ ]:
retriever = HybridRetriever()

# Process the text content
retriever.process(
    content_input_path=text_content,
    chunk_size=500,
)

print("Retriver is ready with embedded chunks")

## Step 3: Retrieve relevant context

Query the retriever with a question about the content.

In [ ]:
query = "What research areas is KAUST known for?"
retrieved_info = retriever.query(
    query=query,
    top_k=3,
    vector_retriever_top_k=5,
    bm25_retriever_top_k=5,
)

for i, chunk in enumerate(retrieved_info):
    print(f"\n--- Result {i+1} ---")
    print(chunk[:300] if len(chunk) > 300 else chunk)

## Step 4: Answer with multimodal context

Pass both the retrieved text and the image to a multimodal model (GPT-4o).

In [ ]:
model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI,
    model_type=ModelType.GPT_4O,
)

agent = ChatAgent(
    system_message="You are a helpful research assistant. Use the retrieved context "
                   "and images to answer questions accurately.",
    model=model,
)

# Combine retrieved text with image context
user_message = f"""Based on the retrieved information and the image provided,
answer this question: {query}

Retrieved context:
{' '.join(str(r) for r in retrieved_info)}"""

response = agent.step(user_message)
print(response.msgs[0].content)

## Summary

You've built a multimodal RAG pipeline that:
1. Loads text and images from URLs
2. Indexes text with a hybrid retriever (vector + BM25)
3. Retrieves relevant chunks for a query
4. Answers using a multimodal model (GPT-4o) with both text and image context

This pattern extends to any document set with associated images â€”
diagrams, charts, screenshots, or photographs.